# 🚀 Dashboard de Investimentos B3
### Célula 1 — Setup: Mock numba + Imports

In [5]:
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# ── Mock numba (não instalado no Termux) ─────────────────────────────────────
# pandas_ta importa numba diretamente em _math.py.
# njit precisa ser um decorator transparente (retorna a própria função).
if 'numba' not in sys.modules:
    from unittest.mock import MagicMock
    _mock = MagicMock()
    _mock.njit = lambda f=None, **kw: (f if f else lambda fn: fn)
    _mock.prange = range
    sys.modules['numba']                  = _mock
    sys.modules['numba.core']             = MagicMock()
    sys.modules['numba.typed']            = MagicMock()
    sys.modules['numba.np']               = MagicMock()
    sys.modules['numba.np.numpy_support'] = MagicMock()
    print('✅ Mock numba aplicado')
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import plotly.graph_objs as go
import plotly.io as pio
from plotly.subplots import make_subplots
from datetime import datetime
from IPython.display import display, HTML

# Módulos do projeto
from dados_financeiros import ColetorDados
from analise_fundamentalista import AnaliseFundamentalista
from analise_tecnica import AnaliseTecnica
from gestao_carteira import GestaoCarteira

pio.renderers.default = 'notebook'  # Renderiza gráficos inline no Jupyter
print('✅ Imports concluídos')

2026-05-15 05:22:10,259 [logging.log_init] INFO: LOGLEVEL=INFO


✅ Imports concluídos


### Célula 2 — Inicialização: Coletor de Dados + Fundamentalista

In [6]:
print('🔄 Inicializando coletor de dados...')
coletor = ColetorDados()

print('🔄 Carregando dados fundamentalistas (pode demorar ~10s)...')
dados_fund = coletor.obter_dados_fundamentalistas()

if dados_fund is not None and not dados_fund.empty:
    analise_fund = AnaliseFundamentalista(dados_fund)
    print(f'✅ {len(dados_fund)} empresas carregadas.')
    display(dados_fund.head(3))  # Preview para confirmar estrutura
else:
    analise_fund = None
    dados_fund = pd.DataFrame()
    print('⚠️ Dados fundamentalistas indisponíveis. Análises fundamentalistas desabilitadas.')

🔄 Inicializando coletor de dados...
✅ Coletor de Dados inicializado com Sessão, Cache e Retentativas.
🔄 Carregando dados fundamentalistas (pode demorar ~10s)...
✅ 973 empresas carregadas.


Multiples,cotacao,pl,pvp,psr,dy,pa,pcg,pebit,pacl,evebit,evebitda,mrgebit,mrgliq,roic,roe,liqc,liq2m,patrliq,divbpatr,c5y
papel,,,,,,,,,,,,,,,,,,,,
AALR3,4.97,-10.79,0.70,0.609,0.0000,0.265,-60.02,6.12,-0.75,10.52,5.62,0.0994,-0.0495,0.0474,-0.0653,0.98,276990.0,1.074970e+09,0.62,0.0532
ABCB3,0.00,0.00,0.00,0.000,0.0000,0.000,0.00,0.00,0.00,0.00,0.00,0.0000,0.0000,0.0000,0.1505,0.00,0.0,6.472310e+09,0.00,0.3058
ABCB4,22.39,5.63,0.85,0.000,0.0869,0.000,0.00,0.00,0.00,0.00,0.00,0.0000,0.0000,0.0000,0.1505,0.00,12687200.0,6.472310e+09,0.00,0.3058


### Célula 3 — Carteira do Usuário

In [8]:
from gestao_carteira import GestaoCarteira

carteira_data = {
    'COGN3': {'qtd': 470,  'preco_compra': 27.90},
    'EGIE3': {'qtd': 200,  'preco_compra': 40.30},
    'BBAS3': {'qtd': 300,  'preco_compra': 22.90},
    'VALE3': {'qtd': 100,  'preco_compra': 50.10},
    'HBOR3': {'qtd': 1500, 'preco_compra':  2.96},
    'IRBR3': {'qtd': 1213, 'preco_compra':  3.17},
    'SIMH3': {'qtd': 600,  'preco_compra':  4.50},
    'AGRO3': {'qtd': 100,  'preco_compra': 20.00},
    'VAMO3': {'qtd': 400,  'preco_compra':  4.65},
    'GFSA3': {'qtd': 600,  'preco_compra':  2.50},
    'EZTC3': {'qtd': 300,  'preco_compra':  4.60},
    'VIVT3': {'qtd': 800,  'preco_compra':  1.50},
    'AMOB3': {'qtd': 450,  'preco_compra':  1.43},
}

carteira_usuario = GestaoCarteira()
for ticker, d in carteira_data.items():
    carteira_usuario.adicionar_ativo(ticker, d['qtd'], d['preco_compra'])

print(f'✅ Carteira carregada com {len(carteira_data)} ativos.')
print('Tickers:', ', '.join(carteira_data.keys()))

✅ Carteira carregada com 13 ativos.
Tickers: COGN3, EGIE3, BBAS3, VALE3, HBOR3, IRBR3, SIMH3, AGRO3, VAMO3, GFSA3, EZTC3, VIVT3, AMOB3


In [16]:
import yfinance as yf
import requests
import requests_cache
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

# Sessão com retry automático e User-Agent de navegador
session = requests.Session()
session.headers.update({
            "User-Agent": (
                "Mozilla/5.0 (Linux; Android 12; Pixel 6) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0 Mobile Safari/537.36"
            )
        })

retry = Retry(
            total=3,
            backoff_factor=0.5,
            status_forcelist=[429, 500, 502, 503, 504],
        )
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

data = yf.download(
                    'COGN3.SA',
                    period="2d",
                    auto_adjust=True,
                    progress=False,
                    group_by="ticker",
                    session=session,
                  )

                   

2026-05-15 07:10:50,973 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'COGN3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 07:10:54,633 [multi.download] ERROR: 
1 Failed download:
2026-05-15 07:10:54,636 [multi.download] ERROR: ['COGN3.SA']: RetryError(MaxRetryError("HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))"))


In [ ]:
print(data)

### Célula 4 — Resumo da Carteira (Dashboard Principal)

In [11]:
import dados_financeiros

print('🔄 Buscando preços atuais...')
precos_atuais = coletor.obter_precos_em_lote(carteira_data.keys())
performance   = carteira_usuario.calcular_performance(precos_atuais)

total_investido = performance.get('TOTAL', {}).get('valor_investido', 0)
valor_atual     = performance.get('TOTAL', {}).get('valor_atual', 0)
lucro           = valor_atual - total_investido
rentabilidade   = (lucro / total_investido * 100) if total_investido else 0

dy_carteira = 0
if analise_fund is not None:
    dy_carteira = carteira_usuario.calcular_dividend_yield_carteira(dados_fund)

2026-05-15 06:45:29,728 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'VIVT3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:45:33,338 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'VAMO3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:45:37,003 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'AMOB3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:45:40,559 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'SIMH3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/get

⚠️ Download em lote vazio na tentativa 2.
⏳ Lote — tentativa 3/3, aguardando 5.8s...


2026-05-15 06:48:35,703 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'VALE3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:48:39,346 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'EGIE3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:48:43,028 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'EZTC3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:48:46,680 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'VIVT3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/get

⚠️ Download em lote vazio na tentativa 3.
❌ Falha após 3 tentativas. Retornando zeros.


KeyError: "None of [Index(['Div.Yield'], dtype='str', name='Multiples')] are in the [columns]"

In [10]:
cor_lucro = '#42b72a' if lucro >= 0 else '#fa3e3e'
sinal     = '+' if lucro >= 0 else ''

display(HTML(f'''
<div style='font-family:sans-serif; display:flex; gap:16px; flex-wrap:wrap;'>
  <div style='background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1); min-width:200px;'>
    <h3 style='color:#1877f2; margin:0 0 12px'>💰 Valor Investido</h3>
    <p style='font-size:24px; font-weight:bold; margin:0'>R$ {total_investido:,.2f}</p>
  </div>
  <div style='background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1); min-width:200px;'>
    <h3 style='color:#1877f2; margin:0 0 12px'>📈 Valor Atual</h3>
    <p style='font-size:24px; font-weight:bold; margin:0'>R$ {valor_atual:,.2f}</p>
  </div>
  <div style='background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1); min-width:200px;'>
    <h3 style='color:#1877f2; margin:0 0 12px'>🎯 Resultado</h3>
    <p style='font-size:24px; font-weight:bold; margin:0; color:{cor_lucro}'>{sinal}R$ {lucro:,.2f} ({sinal}{rentabilidade:.2f}%)</p>
  </div>
  <div style='background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1); min-width:200px;'>
    <h3 style='color:#1877f2; margin:0 0 12px'>💸 DY Médio Carteira</h3>
    <p style='font-size:24px; font-weight:bold; margin:0'>{dy_carteira:.2f}%</p>
  </div>
</div>
'''))

### Célula 5 — 🏆 Ranking Dividend Yield (Top 20)

In [19]:
# Célula — Ranking Dividend Yield
if analise_fund:
    ranking_df = analise_fund.ranking_dividend_yield(20)

    if not ranking_df.empty:
        # Formatos dinâmicos: só aplica formato se a coluna existir
        fmt = {}
        if "Cotação"   in ranking_df.columns: fmt["Cotação"]   = "R$ {:.2f}"
        if "DY"        in ranking_df.columns: fmt["DY"]        = "{:.2%}"
        if "ROE"       in ranking_df.columns: fmt["ROE"]       = "{:.2%}"
        if "P/L"       in ranking_df.columns: fmt["P/L"]       = "{:.2f}"
        if "P/VP"      in ranking_df.columns: fmt["P/VP"]      = "{:.2f}"
        if "Mrg EBIT"  in ranking_df.columns: fmt["Mrg EBIT"]  = "{:.2%}"
        if "Mrg Liq"   in ranking_df.columns: fmt["Mrg Liq"]   = "{:.2%}"

        display(ranking_df.style.format(fmt))
    else:
        print("⚠️ Ranking vazio — nenhuma empresa passou nos filtros.")
else:
    print("❌ Dados fundamentalistas indisponíveis.")

Multiples,Cotação,DY,ROE,P/L,P/VP,Mrg EBIT,Mrg Liq,Patrim Liq
papel,,,,,,,,
SYNE3,R$ 6.53,107.35%,51.13%,1.79,0.91,35.33%,50.48%,1091940000.000000
TRPN3,R$ 0.71,104.74%,122.35%,0.44,0.54,72.28%,59.29%,59358000.000000
BSLI4,R$ 8.56,48.36%,5.31%,21.26,1.13,0.00%,0.00%,3687330000.000000
BSLI3,R$ 8.03,46.87%,5.31%,19.95,1.06,0.00%,0.00%,3687330000.000000
JBSS3,R$ 39.03,39.27%,25.15%,3.11,0.78,5.86%,2.75%,43308500000.000000
EPAR3,R$ 4.96,37.04%,10.50%,12.84,1.35,-91.39%,81.38%,54645000.000000
MELK3,R$ 3.37,34.85%,5.03%,12.76,0.64,6.31%,11.46%,1083040000.000000
MMAQ4,R$ 2550.01,24.28%,14.45%,2.28,0.33,-0.94%,3.21%,286268000.000000
PATI4,R$ 34.76,23.79%,11.05%,9.57,1.06,5.62%,3.99%,786260000.000000


### Célula 6 — 🎯 Filtro Método Bazin

In [21]:
if analise_fund:
    bazin_df = analise_fund.filtrar_metodo_bazin()
    print(f'✅ {len(bazin_df)} empresas aprovadas no Método Bazin')

    if not bazin_df.empty:
        # Formatos dinâmicos — só aplica se a coluna existir
        fmt = {}
        if 'Cotação'    in bazin_df.columns: fmt['Cotação']    = 'R$ {:.2f}'
        if 'DY'         in bazin_df.columns: fmt['DY']         = '{:.2f}%'
        if 'ROE'        in bazin_df.columns: fmt['ROE']        = '{:.2%}'
        if 'Mrg EBIT'   in bazin_df.columns: fmt['Mrg EBIT']   = '{:.2%}'
        if 'Dív/Patrim' in bazin_df.columns: fmt['Dív/Patrim'] = '{:.2f}'

        display(
            bazin_df.style
            .format(fmt)
            .background_gradient(
                subset=['DY'] if 'DY' in bazin_df.columns else None,
                cmap='Greens'
            )
            .set_caption('Empresas aprovadas — Método Décio Bazin')
        )
    else:
        print('⚠️ Nenhuma empresa passou nos critérios do Método Bazin.')
else:
    print('⚠️ Dados fundamentalistas indisponíveis')

✅ 23 empresas aprovadas no Método Bazin


Multiples,Cotação,DY,ROE,Mrg EBIT,Dív/Patrim
papel,,,,,
TRPN3,R$ 0.71,1.05%,122.35%,72.28%,0.00
SOND6,R$ 37.85,0.17%,33.27%,15.66%,0.00
RECV3,R$ 14.33,0.15%,12.43%,30.18%,0.42
SOND5,R$ 44.00,0.14%,33.27%,15.66%,0.00
SOND3,R$ 47.00,0.12%,33.27%,15.66%,0.00
CGRA4,R$ 28.58,0.11%,10.93%,11.74%,0.00
CGRA3,R$ 28.98,0.11%,10.93%,11.74%,0.00
VLID3,R$ 25.61,0.11%,17.51%,17.37%,0.34
CEBR6,R$ 21.05,0.10%,16.18%,33.67%,0.00


### Célula 7 — 📈 Análise Técnica
> Altere `TICKER` na linha abaixo e re-execute a célula.

In [17]:
# ── Fonte alternativa: brapi.dev (fallback quando Yahoo está bloqueado) ──────
import requests as _requests

def obter_historico_brapi(ticker: str, periodo: str = "1y") -> pd.DataFrame:
    """
    Busca histórico OHLCV via brapi.dev.
    Não precisa de API key. Cobre todos os tickers da B3.
    Períodos aceitos: 1d 5d 1mo 3mo 6mo 1y 2y 5y 10y ytd max
    """
    periodo_map = {
        "3mo": "3mo", "6mo": "6mo",
        "1y":  "1y",  "2y":  "2y",
    }
    range_param = periodo_map.get(periodo, "1y")
    url = (
        f"https://brapi.dev/api/quote/{ticker}"
        f"?range={range_param}&interval=1d&fundamental=false"
    )

    try:
        resp = _requests.get(url, timeout=20)
        resp.raise_for_status()
        data = resp.json()

        results = data.get("results", [])
        if not results:
            print(f"⚠️ brapi: sem resultados para {ticker}")
            return pd.DataFrame()

        hist = results[0].get("historicalDataPrice", [])
        if not hist:
            print(f"⚠️ brapi: sem histórico para {ticker}")
            return pd.DataFrame()

        df = pd.DataFrame(hist)

        # timestamp Unix → datetime
        df["date"] = pd.to_datetime(df["date"], unit="s", utc=True).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)
        df = df.set_index("date")
        df.index.name = "Date"

        df = df.rename(columns={
            "open":   "Open",
            "high":   "High",
            "low":    "Low",
            "close":  "Close",
            "volume": "Volume",
        })

        colunas = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in df.columns]
        df = df[colunas].dropna(subset=["Close"])
        df = df.sort_index()

        print(f"✅ brapi: {len(df)} candles carregados para {ticker}")
        return df

    except Exception as e:
        print(f"🚨 Erro brapi ({ticker}): {e}")
        return pd.DataFrame()


print("✅ Fonte alternativa brapi.dev pronta.")

✅ Fonte alternativa brapi.dev pronta.


In [18]:
# ── Configuração ─────────────────────────────
TICKER  = 'GRND3'   # <- altere aqui
PERIODO = '1y'       # 3mo | 6mo | 1y | 2y
# ─────────────────────────────────────────────

print(f'🔄 Buscando histórico de {TICKER}...')
#dados_hist = coletor.obter_historico_preco(TICKER, PERIODO)
dados_hist = obter_historico_brapi(TICKER, PERIODO)  
if dados_hist.empty:
    print(f'❌ Nenhum dado encontrado para {TICKER}. Verifique o código do ativo.')
else:
    analise_tec              = AnaliseTecnica(dados_hist)
    dados_ind                = analise_tec.calcular_indicadores()
    sinais                   = analise_tec.identificar_sinais()
    suporte, resistencia     = analise_tec.calcular_suporte_resistencia()

    # ── Gráfico principal (candlestick + médias) ──────────────────────────────
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        row_heights=[0.6, 0.2, 0.2],
        subplot_titles=(f'{TICKER} — Preço', 'RSI (14)', 'MACD')
    )

    # Candlestick
    fig.add_trace(go.Candlestick(
        x=dados_ind.index,
        open=dados_ind['Open'], high=dados_ind['High'],
        low=dados_ind['Low'],   close=dados_ind['Close'],
        name='Preço', showlegend=False
    ), row=1, col=1)

    # Médias móveis
    for col, cor, nome in [('SMA_20','blue','SMA 20'), ('SMA_50','orange','SMA 50'), ('BB_Upper','gray','BB +'), ('BB_Lower','gray','BB -')]:
        if col in dados_ind.columns:
            fig.add_trace(go.Scatter(
                x=dados_ind.index, y=dados_ind[col],
                name=nome, line=dict(color=cor, width=1, dash='dot' if 'BB' in col else 'solid')
            ), row=1, col=1)

    # Suporte / Resistência
    if suporte:    fig.add_hline(y=suporte,    line_dash='dash', line_color='green', row=1, col=1)
    if resistencia: fig.add_hline(y=resistencia, line_dash='dash', line_color='red',   row=1, col=1)

    # RSI
    if 'RSI' in dados_ind.columns:
        fig.add_trace(go.Scatter(x=dados_ind.index, y=dados_ind['RSI'], name='RSI', line=dict(color='purple', width=1)), row=2, col=1)
        fig.add_hline(y=70, line_dash='dash', line_color='red',   row=2, col=1)
        fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)

    # MACD
    if 'MACD' in dados_ind.columns:
        fig.add_trace(go.Scatter(x=dados_ind.index, y=dados_ind['MACD'],        name='MACD',   line=dict(color='blue', width=1)), row=3, col=1)
        fig.add_trace(go.Scatter(x=dados_ind.index, y=dados_ind['MACD_Signal'], name='Signal', line=dict(color='orange', width=1)), row=3, col=1)
        fig.add_bar(x=dados_ind.index, y=dados_ind['MACD_Histogram'], name='Histograma', row=3, col=1)

    fig.update_layout(
        height=700, title=f'Análise Técnica — {TICKER}',
        xaxis_rangeslider_visible=False,
        legend=dict(orientation='h', yanchor='bottom', y=1.02)
    )
    fig.show()

    # ── Sinais e níveis ────────────────────────────────────────────────────────
    sinais_txt = '<br>'.join([
        f"<span style='color:{'#42b72a' if 'COMPRA' in s else '#fa3e3e'}'>● {s}</span>"
        for s in sinais
    ]) or '<span style="color:#999">Nenhum sinal claro identificado.</span>'

    display(HTML(f'''
    <div style='font-family:sans-serif; background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1); margin-top:12px;'>
      <h3 style='color:#1877f2'>🎯 Sinais Identificados</h3>
      <p>{sinais_txt}</p>
      <h3 style='color:#1877f2'>📊 Níveis Chave</h3>
      <p><strong>Suporte:</strong> R$ {suporte:.2f if suporte else 'N/A'}</p>
      <p><strong>Resistência:</strong> R$ {resistencia:.2f if resistencia else 'N/A'}</p>
    </div>
    '''))

🔄 Buscando histórico de GRND3...
🚨 Erro brapi (GRND3): 401 Client Error: Unauthorized for url: https://brapi.dev/api/quote/GRND3?range=1y&interval=1d&fundamental=false
❌ Nenhum dado encontrado para GRND3. Verifique o código do ativo.


### Célula 8 — ⏰ Monitoramento da Carteira em Tempo Real

In [ ]:
print('🔄 Buscando dados em tempo real...')
dados_tr = coletor.monitorar_carteira_tempo_real(list(carteira_data.keys()))

rows = ''
for ticker, d in dados_tr.items():
    cor  = '#42b72a' if d['variacao_dia'] >= 0 else '#fa3e3e'
    sinal = '+' if d['variacao_dia'] >= 0 else ''
    rows += f"""
    <tr>
      <td><strong>{ticker}</strong></td>
      <td>R$ {d['preco']:.2f}</td>
      <td style='color:{cor}; font-weight:bold'>{sinal}{d['variacao_dia']:.2f}%</td>
      <td>{d['volume']:,.0f}</td>
    </tr>"""

display(HTML(f'''
<div style='font-family:sans-serif; background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1);'>
  <h3 style='color:#1877f2'>⏰ Preços em Tempo Real — {datetime.now().strftime('%d/%m/%Y %H:%M')}</h3>
  <table style='width:100%; border-collapse:collapse;'>
    <thead>
      <tr style='background:#f5f7fa;'>
        <th style='padding:10px; text-align:left;'>Ticker</th>
        <th style='padding:10px; text-align:left;'>Preço</th>
        <th style='padding:10px; text-align:left;'>Var. Dia</th>
        <th style='padding:10px; text-align:left;'>Volume</th>
      </tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
</div>
'''))

### Célula 9 — 💼 Performance Detalhada da Carteira

In [ ]:
# Reutiliza precos_atuais da Célula 4 (re-execute a Célula 4 para atualizar)
perf_rows = ''
for ticker, d in performance.items():
    if ticker == 'TOTAL':
        continue
    cor   = '#42b72a' if d['lucro_prejuizo'] >= 0 else '#fa3e3e'
    sinal = '+' if d['lucro_prejuizo'] >= 0 else ''
    perf_rows += f"""
    <tr>
      <td><strong>{ticker}</strong></td>
      <td>{d['quantidade']}</td>
      <td>R$ {d['preco_compra']:.2f}</td>
      <td>R$ {d['preco_atual']:.2f}</td>
      <td>R$ {d['valor_investido']:,.2f}</td>
      <td>R$ {d['valor_atual']:,.2f}</td>
      <td style='color:{cor}; font-weight:bold'>{sinal}R$ {d['lucro_prejuizo']:,.2f}<br><small>{sinal}{d['rentabilidade']:.2f}%</small></td>
    </tr>"""

total     = performance.get('TOTAL', {})
cor_total = '#42b72a' if total.get('lucro_prejuizo', 0) >= 0 else '#fa3e3e'
sinal_tot = '+' if total.get('lucro_prejuizo', 0) >= 0 else ''

display(HTML(f'''
<div style='font-family:sans-serif; background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1);'>
  <h3 style='color:#1877f2'>💼 Performance da Carteira</h3>
  <table style='width:100%; border-collapse:collapse;'>
    <thead>
      <tr style='background:#f5f7fa; font-weight:600;'>
        <th style='padding:10px; text-align:left;'>Ticker</th>
        <th style='padding:10px; text-align:left;'>Qtd</th>
        <th style='padding:10px; text-align:left;'>Preço Médio</th>
        <th style='padding:10px; text-align:left;'>Preço Atual</th>
        <th style='padding:10px; text-align:left;'>Investido</th>
        <th style='padding:10px; text-align:left;'>Atual</th>
        <th style='padding:10px; text-align:left;'>Resultado</th>
      </tr>
    </thead>
    <tbody>
      {perf_rows}
      <tr style='background:#f5f7fa; font-weight:bold;'>
        <td colspan='4'>TOTAL</td>
        <td>R$ {total.get('valor_investido',0):,.2f}</td>
        <td>R$ {total.get('valor_atual',0):,.2f}</td>
        <td style='color:{cor_total}'>{sinal_tot}R$ {total.get('lucro_prejuizo',0):,.2f} ({sinal_tot}{total.get('rentabilidade',0):.2f}%)</td>
      </tr>
    </tbody>
  </table>
</div>
'''))